# FedTrace — 01: Data Source Exploration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fedtrace/fedtrace.github.io/blob/main/notebooks/01_data_sources.ipynb)

**Runtime:** CPU only  
**Estimated time:** ~20 min  
**Publishes:** `data/outputs/01_data_sources.json`, `data/findings/01_data_sources.md`

**Driving questions:**
1. For any cancelled federal award, what are the three numbers that define its financial record — ceiling, obligated, outlays — and how reliably can they be reconstructed from public sources?
2. For each DOGE PIID: is it a direct contract award or an IDV vehicle? The answer determines where obligation data lives.
3. What fraction of cancelled contracts can be joined to USASpending records, and what structural factors explain the gaps?
4. For cancelled grants, does the DOGE `link` field provide a reliable path to the full award record?

Each section below responds to one or more driving questions with evidence from current data. Findings state what the record shows. Interpretations are marked provisional — the same evidence may read differently as coverage expands. Current findings: [`data/findings/01_data_sources.md`](../data/findings/01_data_sources.md).

In [ ]:
# ── CELL 1: Clone repo & install dependencies ─────────────────────────────────
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/fedtrace.github.io')
if not REPO.exists():
    env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    subprocess.run(
        ['git', 'clone', '--depth=1', 'https://github.com/fedtrace/fedtrace.github.io.git', str(REPO)],
        check=True, env=env,
    )
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import ensure_notebook_requirements
ensure_notebook_requirements('01_data_sources', requirements_path=str(REPO / 'requirements.txt'))

In [ ]:
# ── CELL 2: Pull latest & set up paths ────────────────────────────────────────
from scripts.colab_utils import prepare_notebook
REPO, PATHS = prepare_notebook(REPO, pull_latest=True)
PATHS['outputs_dir'].mkdir(parents=True, exist_ok=True)
print(f'Repo ready at {REPO}')

In [ ]:
import requests
import pandas as pd
import json
import re
import time
import urllib.parse

pd.set_option('display.max_colwidth', 80)

DOGE_BASE = 'https://api.doge.gov'
USA_BASE  = 'https://api.usaspending.gov/api/v2'

# Award type codes
CONTRACT_TYPES = ['A', 'B', 'C', 'D']
IDV_TYPES      = ['IDV_A', 'IDV_B', 'IDV_B_A', 'IDV_B_B', 'IDV_B_C', 'IDV_C', 'IDV_D', 'IDV_E']

USA_FIELDS = [
    'Award ID', 'Recipient Name', 'Award Amount', 'Total Outlays',
    'Description', 'Contract Award Type', 'Awarding Agency',
    'Period of Performance Start Date',
    'Period of Performance Current End Date',
    'generated_internal_id',
]

def doge_fetch(endpoint, per_page=500, max_pages=None):
    records, page = [], 1
    while True:
        r = requests.get(f'{DOGE_BASE}{endpoint}', params={'per_page': per_page, 'page': page})
        r.raise_for_status()
        data = r.json()
        key = next(iter(data['result']))
        records.extend(data['result'][key])
        total_pages = data['meta']['pages']
        print(f'  page {page}/{total_pages} — {len(records):,}', end='\r')
        if page >= total_pages or (max_pages and page >= max_pages):
            break
        page += 1
        time.sleep(0.1)
    print()
    return pd.DataFrame(records)

## 1. DOGE — Contracts & Grants

In [ ]:
print('Fetching contracts...')
contracts = doge_fetch('/savings/contracts')
print(f'Contracts: {len(contracts):,}')

print('Fetching grants...')
grants = doge_fetch('/savings/grants')
print(f'Grants: {len(grants):,}')

In [ ]:
# Classify PIIDs upfront: single resolvable vs aggregate bundles
contracts['piid_type'] = contracts['piid'].apply(
    lambda x: 'aggregate' if str(x).startswith('Multiple') or ' ' in str(x) else 'single'
)
n_single = (contracts['piid_type'] == 'single').sum()
n_agg    = (contracts['piid_type'] == 'aggregate').sum()
print(f'Single PIIDs: {n_single:,} | Aggregate bundles: {n_agg:,}')
print()
print('Null rates:')
print((contracts.isnull().sum() / len(contracts) * 100).round(1).to_string())

## 2. DOGE Payments

In [ ]:
print('Fetching payments sample (5 pages)...')
payments = doge_fetch('/payments', max_pages=5)
payments['justification_len'] = payments['recipient_justification'].fillna('').str.len()
print(f'Sample: {len(payments):,} | FAIN coverage: {payments["fain"].notna().mean()*100:.1f}%')

## 3. USASpending Join — Two-Pass with IDV Resolution

**Why two passes:**  
A DOGE PIID may be a direct contract award (obligations on the record itself) or an IDV vehicle (obligations live in child task orders, the IDV record carries $0). A single query with contract type codes misses IDVs entirely — they appear as unmatched.

**Pass 1:** query as contracts (`award_type_codes: A,B,C,D`)  
**Pass 2:** for unmatched PIIDs, query as IDVs (`IDV_A` through `IDV_E`)  
**IDV resolution:** for each matched IDV, call `/api/v2/idvs/amounts/{id}/` to get aggregated child task order obligations — the actual spend.

In [ ]:
def _usa_search(piids, award_type_codes):
    """Single-pass USASpending search for a list of PIIDs and award types."""
    results = []
    for i in range(0, len(piids), 50):
        batch = piids[i:i+50]
        payload = {
            'filters': {'award_ids': batch, 'award_type_codes': award_type_codes},
            'fields': USA_FIELDS,
            'page': 1, 'limit': 50,
        }
        r = requests.post(f'{USA_BASE}/search/spending_by_award/', json=payload)
        r.raise_for_status()
        results.extend(r.json()['results'])
        time.sleep(0.2)
    return pd.DataFrame(results) if results else pd.DataFrame(columns=USA_FIELDS)


def get_idv_amounts(generated_internal_id):
    """Return child task order obligations and outlays for an IDV.

    The IDV record carries $0 in obligations. Actual spend flows through child
    task orders. The correct fields are child_award_total_obligation and
    child_award_total_outlay — not obligated_amount, which does not exist.

    Returns (child_obligation, child_outlay) or (None, None) on failure.
    """
    try:
        r = requests.get(f'{USA_BASE}/idvs/amounts/{generated_internal_id}/', timeout=10)
        if r.status_code == 200:
            body = r.json()
            return (
                body.get('child_award_total_obligation'),
                body.get('child_award_total_outlay'),
            )
    except Exception:
        pass
    return None, None


def lookup_two_pass(piids):
    """Resolve PIIDs to USASpending records with IDV hierarchy awareness.

    Returns DataFrame with columns from USA_FIELDS plus:
      record_type: 'contract' | 'idv'
      child_obligations: float | None  (IDVs only — child_award_total_obligation)
      child_outlays:     float | None  (IDVs only — child_award_total_outlay)
    """
    # Pass 1 — direct contract awards
    print('Pass 1: contract types...')
    contracts_df = _usa_search(piids, CONTRACT_TYPES)
    contracts_df['record_type'] = 'contract'
    contracts_df['child_obligations'] = None
    contracts_df['child_outlays'] = None
    matched_piids = set(contracts_df['Award ID'].tolist())
    print(f'  matched {len(matched_piids):,} as contracts')

    # Pass 2 — IDV vehicles for unmatched
    unmatched = [p for p in piids if p not in matched_piids]
    idv_df = pd.DataFrame(columns=USA_FIELDS)
    if unmatched:
        print(f'Pass 2: IDV types for {len(unmatched):,} unmatched...')
        idv_df = _usa_search(unmatched, IDV_TYPES)
        idv_df['record_type'] = 'idv'
        idv_df['child_obligations'] = None
        idv_df['child_outlays'] = None
        print(f'  matched {len(idv_df):,} as IDVs')

        if len(idv_df) > 0 and 'generated_internal_id' in idv_df.columns:
            print('  fetching IDV child amounts...')
            for idx, row in idv_df.iterrows():
                gid = row.get('generated_internal_id')
                if gid:
                    obl, out = get_idv_amounts(gid)
                    idv_df.at[idx, 'child_obligations'] = obl
                    idv_df.at[idx, 'child_outlays'] = out
                    time.sleep(0.1)

    combined = pd.concat([contracts_df, idv_df], ignore_index=True)
    still_unmatched = len(piids) - len(set(combined['Award ID'].tolist()))
    print(f'Still unmatched after both passes: {still_unmatched:,}')
    return combined

In [ ]:
# Stratified sample across value distribution
single = contracts[contracts['piid_type'] == 'single'].copy()
sample = pd.concat([
    single.nlargest(50, 'value'),
    single.sample(100, random_state=42),
    single.nsmallest(50, 'value'),
]).drop_duplicates(subset='piid')
piids = sample['piid'].tolist()
print(f'Sample: {len(sample)} contracts')

usa = lookup_two_pass(piids)

In [ ]:
print('Record type breakdown:')
print(usa['record_type'].value_counts().to_string())
print()

# Dedup within each pass (fan-out from modifications)
usa_dedup = (
    usa.sort_values('Award Amount', ascending=False)
    .groupby('Award ID', as_index=False)
    .first()
)

joined = sample.merge(usa_dedup, left_on='piid', right_on='Award ID', how='left')
matched   = joined['Award ID'].notna()
contracts_matched = matched & (joined['record_type'] == 'contract')
idv_matched       = matched & (joined['record_type'] == 'idv')

print(f'Total match rate:     {matched.mean()*100:.1f}% ({matched.sum()}/{len(joined)})')
print(f'  — as contracts:     {contracts_matched.sum()}')
print(f'  — as IDVs:          {idv_matched.sum()}')
print(f'  — unmatched:        {(~matched).sum()}')
print()
print('Unmatched PIIDs (first 10):')
print(joined[~matched]['piid'].head(10).tolist())

## 3a. Diagnostics — Unmatched PIIDs & IDV Resolution

Two open questions from the join above, investigated before building at scale.

In [ ]:
# Diagnose unmatched PIIDs — understand structural vs fixable gaps
unmatched_df = joined[~matched][['piid', 'agency', 'vendor', 'value', 'savings']].copy()
print(f'Unmatched: {len(unmatched_df)}')
print()

# Sentinel values that can never resolve
sentinel = unmatched_df[
    unmatched_df['piid'].isin(['Unavailable', '']) |
    unmatched_df['piid'].str.startswith('Multiple', na=False)
]
print(f'Sentinel PIIDs (Unavailable / Multiple): {len(sentinel)}')
print()

# Agency distribution — unmatched vs matched
print('Agency breakdown (top 10):')
agency_comp = pd.DataFrame({
    'unmatched': unmatched_df['agency'].value_counts().head(10),
    'matched':   joined[matched]['agency'].value_counts().head(10),
}).fillna(0).astype(int)
print(agency_comp.to_string())
print()

# Value distribution — are unmatched awards disproportionately large or small?
print('Value distribution — unmatched:')
print(unmatched_df['value'].describe().apply(lambda x: f'${x:,.0f}').to_string())
print()

print('Unmatched PIIDs (sample 20):')
print(unmatched_df['piid'].tolist()[:20])

In [ ]:
# Diagnose IDV child obligation null returns
# Show raw generated_internal_id values and exact API responses.
idv_rows = usa[usa['record_type'] == 'idv'].copy()
print(f'IDV rows: {len(idv_rows)}')
print()

if len(idv_rows) > 0:
    for _, row in idv_rows.head(5).iterrows():
        gid = row.get('generated_internal_id')
        print(f'PIID: {row.get("Award ID")}')
        print(f'  generated_internal_id: {repr(gid)}')
        if gid:
            url = f'{USA_BASE}/idvs/amounts/{gid}/'
            r = requests.get(url, timeout=10)
            print(f'  status: {r.status_code}')
            if r.status_code == 200:
                body = r.json()
                print(f'  keys: {list(body.keys())}')
                print(f'  child_idv_count: {body.get("child_idv_count")}')
                print(f'  child_award_count: {body.get("child_award_count")}')
                print(f'  obligated_amount: {body.get("obligated_amount")}')
                print(f'  total_obligated_amount: {body.get("total_obligated_amount")}')
            else:
                print(f'  body: {r.text[:300]}')
        print()

## 4. Three-Number Record

For every cancelled award, the accountability record needs three numbers:
- **Ceiling** — DOGE `value` (FPDS Base and All Options Value, including unexercised options)
- **Obligated** — for contracts: USASpending `Award Amount`; for IDVs: sum of child task order obligations
- **Outlays** — USASpending `Total Outlays` (actual disbursements)

DOGE `savings` = ceiling − obligated. It is unexercised headroom, not recovered money.

In [ ]:
df = joined[matched].copy()

# Obligated: Award Amount for direct contracts; child_award_total_obligation for IDVs
df['obligated'] = df.apply(
    lambda r: r['child_obligations'] if r['record_type'] == 'idv' else r['Award Amount'],
    axis=1,
)

# Outlays: Total Outlays for direct contracts; child_award_total_outlay for IDVs
df['outlays'] = df.apply(
    lambda r: r['child_outlays'] if r['record_type'] == 'idv' else r['Total Outlays'],
    axis=1,
)

df['ceiling'] = df['value']   # DOGE value = FPDS ceiling

df['gap_ceiling_to_obligated'] = df['ceiling'] - df['obligated']
df['gap_pct']  = df['gap_ceiling_to_obligated'] / df['ceiling'] * 100
df['pct_saved'] = df['savings'] / df['ceiling'] * 100

print('=== Ceiling vs Obligated gap ===')
print(df['gap_ceiling_to_obligated'].describe().apply(lambda x: f'${x:,.0f}').to_string())
print()
print('Gap as % of ceiling:')
print(df['gap_pct'].describe().round(1).to_string())

In [ ]:
# Gap by record type — contracts vs IDVs
print('Gap by record type:')
print(
    df.groupby('record_type')['gap_pct']
    .agg(['count', 'median', 'mean'])
    .round(1).to_string()
)
print()

# Gap by contract award type
print('Gap by Contract Award Type:')
print(
    df.groupby('Contract Award Type')['gap_pct']
    .agg(['count', 'median'])
    .round(1)
    .sort_values('median', ascending=False)
    .to_string()
)

In [ ]:
# Outlays coverage — how often do we have actual disbursement data?
outlay_coverage = df['outlays'].notna().mean() * 100
print(f'Outlay data available: {outlay_coverage:.1f}% of matched records')
print()

# Three-number record for top 10 by ceiling
cols = ['piid', 'vendor', 'agency', 'ceiling', 'obligated', 'outlays',
        'savings', 'pct_saved', 'record_type']
print('Three-number record — top 10 by ceiling:')
print(df.nlargest(10, 'ceiling')[cols].to_string())

In [ ]:
# Savings field vs obligated amount — relationship check
# DOGE savings = ceiling minus current obligations (a ceiling-based figure).
# USASpending Award Amount = cumulative legally committed spend.
# These measure different things. Where savings > obligated, the ceiling
# headroom exceeds what was ever committed — structurally expected for
# contracts with large unexercised options, but worth surfacing per record.
df['savings_vs_obligated'] = df['savings'] - df['obligated']
savings_exceeds_obligation = df[df['savings_vs_obligated'] > 0]
n_exceeds = len(savings_exceeds_obligation)
print(f'Records where savings field > obligated amount: {n_exceeds} ({n_exceeds/len(df)*100:.1f}%)')
print('(savings = ceiling − obligations; Award Amount = cumulative committed spend)')
if n_exceeds:
    print(savings_exceeds_obligation[['piid', 'vendor', 'ceiling', 'obligated', 'savings']].head(5).to_string())

## 5. Grant Linkage — `link` Field → Direct Award Lookup

The confirmed path (research-validated):  
`link` → extract `generated_unique_award_id` → `GET /api/v2/awards/{id}/`  
Covers ~78% of cancelled grants. No fuzzy matching required.

## 5b. What the Record Shows — and What Remains Open

Current findings are maintained in [`data/findings/01_data_sources.md`](../data/findings/01_data_sources.md) — written automatically by the publish cell on each run. Edit that file, not this section.

In [ ]:
def classify_link(url):
    if not url:
        return 'null'
    try:
        host = urllib.parse.urlparse(str(url)).netloc.lower()
    except Exception:
        return 'unparseable'
    for label in ('usaspending', 'sam.gov', 'grants.gov'):
        if label in host:
            return label
    return host or 'no_host'

def extract_award_id(url):
    """Extract generated_unique_award_id from a USASpending /award/ URL."""
    if not url:
        return None
    m = re.search(r'/award/([^/?#]+)', str(url))
    return m.group(1) if m else None

grants['link_dest'] = grants['link'].apply(classify_link)
print('Link destinations:')
print(grants['link_dest'].value_counts().to_string())

In [ ]:
usa_grants = grants[grants['link_dest'] == 'usaspending'].copy()
usa_grants['award_id'] = usa_grants['link'].apply(extract_award_id)

extractable = usa_grants['award_id'].notna().sum()
print(f'USASpending grant links: {len(usa_grants):,}')
print(f'Award ID extractable:    {extractable:,} ({extractable/len(usa_grants)*100:.1f}%)')
print()
print('Sample extracted IDs:')
print(usa_grants[['recipient', 'award_id']].head(8).to_string())

In [ ]:
# Direct award lookup — one API call per ID, no search ambiguity
def lookup_grant_by_award_id(award_id):
    try:
        r = requests.get(f'{USA_BASE}/awards/{award_id}/', timeout=10)
        if r.status_code == 200:
            d = r.json()
            return {
                'award_id': award_id,
                'recipient': d.get('recipient', {}).get('recipient_name'),
                'award_amount': d.get('total_obligation'),
                'total_outlays': d.get('total_outlays'),
                'awarding_agency': d.get('awarding_agency', {}).get('toptier_agency', {}).get('name'),
                'description': d.get('description'),
                'start_date': d.get('period_of_performance', {}).get('start_date'),
                'end_date': d.get('period_of_performance', {}).get('end_date'),
            }
    except Exception:
        pass
    return {'award_id': award_id, 'error': True}

# Test on first 10
test_ids = usa_grants['award_id'].dropna().head(10).tolist()
grant_results = [lookup_grant_by_award_id(aid) for aid in test_ids]
grant_df = pd.DataFrame(grant_results)

success = grant_df.get('error', pd.Series([False]*len(grant_df))).fillna(False)
print(f'Direct lookup success: {(~success).sum()}/{len(test_ids)}')
print()
print(grant_df[['award_id', 'recipient', 'award_amount', 'awarding_agency']].to_string())

## 6. Publish Results

In [ ]:
output = {
    'doge_contracts_total': len(contracts),
    'doge_contracts_single_piid': int(n_single),
    'doge_contracts_aggregate_piid': int(n_agg),
    'doge_grants_total': len(grants),
    'doge_payments_total_per_api': 107497,
    'usa_match_rate_pct': round(matched.mean() * 100, 1),
    'usa_matched_as_contract': int(contracts_matched.sum()),
    'usa_matched_as_idv': int(idv_matched.sum()),
    'usa_unmatched': int((~matched).sum()),
    'three_number_record': {
        'ceiling_to_obligated_gap_median_usd': round(float(df['gap_ceiling_to_obligated'].median()), 2),
        'ceiling_to_obligated_gap_pct_median': round(float(df['gap_pct'].median()), 1),
        'savings_exceeds_obligation_count': int(n_exceeds),
        'outlay_coverage_pct': round(outlay_coverage, 1),
    },
    'grant_linkage': {
        'link_destinations': grants['link_dest'].value_counts().to_dict(),
        'usaspending_links': int(len(usa_grants)),
        'award_id_extractable': int(extractable),
        'direct_lookup_tested': len(test_ids),
        'direct_lookup_success': int((~success).sum()),
    },
    'join_strategy': {
        'contracts_pass1': 'PIID lookup with award_type_codes A,B,C,D',
        'contracts_pass2': 'unmatched PIIDs with IDV_A..IDV_E types',
        'idv_obligations': 'GET /api/v2/idvs/amounts/{generated_internal_id}/ → child_award_total_obligation',
        'grants': 'extract generated_unique_award_id from link field → GET /api/v2/awards/{id}/',
        'grants_fallback': 'USASpending bulk Assistance_PrimeTransactions + local join',
    },
}

output_path = PATHS['outputs_dir'] / '01_data_sources.json'
output_path.write_text(json.dumps(output, indent=2))
print(json.dumps(output, indent=2))

In [ ]:
from scripts.colab_utils import publish_artifacts

publish_artifacts(
    paths=[
        'data/outputs/01_data_sources.json',
        'data/findings/01_data_sources.md',
    ],
    message='Data source exploration results',
    repo_dir=REPO,
)

In [ ]:
from scripts.colab_utils import publish_artifacts

publish_artifacts(
    paths=['data/outputs/01_data_sources.json'],
    message='Data source exploration results',
    repo_dir=REPO,
)